In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind

In [2]:
df = pd.read_csv("bottlenecks_24_48hpi.csv")
alpha = 0.05
cases = [(24, "Lung"), (48, "Lung"), (48, "Blood")]
results = []

for hpi, tissue in cases:
    subset = df.loc[(df["hpi"] == hpi) & (df["tissue"] == tissue)]
    groups = {}
    for condition in ["Dox", "noDox"]:
        rows = subset.loc[subset["condition"] == condition]
        values = pd.to_numeric(rows["B_hat"], errors="raise")
        valid = np.isfinite(values) & (values > 0) & rows["status"].eq("finite")
        if not valid.all():
            raise ValueError(f"{hpi} hpi {tissue} {condition}: invalid estimates; inspect before testing.")
        if rows["mouse"].duplicated().any():
            raise ValueError("Multiple estimates for the same mouse within a group.")
        groups[condition] = values.to_numpy(dtype=float)
        if len(groups[condition]) < 2:
            raise ValueError("At least two observations per group are required.")

    x, y = groups["Dox"], groups["noDox"]
    mw = mannwhitneyu(x, y, alternative="two-sided", method="auto")
    welch = ttest_ind(x, y, equal_var=False, alternative="two-sided")

    print(f"\n{hpi} hpi — {tissue}")
    for name, values in groups.items():
        print(f"{name} values: {values}")
        print(f"  n={len(values)}, mean={values.mean():.3f}, median={np.median(values):.3f}")
    print(f"Mann–Whitney: U={mw.statistic:.6g}, p={mw.pvalue:.6g}")
    print(f"Welch: t={welch.statistic:.6g}, p={welch.pvalue:.6g}")

    results.append({
        "case": f"{hpi} hpi {tissue}",
        "n_Dox": len(x), "n_noDox": len(y),
        "mean_Dox": x.mean(), "mean_noDox": y.mean(),
        "median_Dox": np.median(x), "median_noDox": np.median(y),
        "MW_U": mw.statistic, "MW_p": mw.pvalue,
        "Welch_t": welch.statistic, "Welch_p": welch.pvalue,
        "MW_p_below_0.05_unadjusted": mw.pvalue < alpha,
        "Welch_p_below_0.05_unadjusted": welch.pvalue < alpha,
    })

results = pd.DataFrame(results)
print("\nSummary (two-sided, unadjusted p-values)")
print(results.to_string(index=False))


24 hpi — Lung
Dox values: [5206. 3576. 4295. 4484. 5418. 3578. 4891. 5832. 4334. 5157.]
  n=10, mean=4677.100, median=4687.500
noDox values: [238145. 281854. 104967. 420125. 422000. 640306.   4032.  21309.   7058.
  28019. 146193.]
  n=11, mean=210364.364, median=146193.000
Mann–Whitney: U=8, p=0.00105875
Welch: t=-3.22428, p=0.00910845

48 hpi — Lung
Dox values: [ 50.  35.  12.  16.  20.  57.  55. 104.  55.]
  n=9, mean=44.889, median=50.000
noDox values: [  27.   11.   10.   33.    9.   20.   36.   26.   42. 1298.]
  n=10, mean=151.200, median=26.500
Mann–Whitney: U=61.5, p=0.191029
Welch: t=-0.83168, p=0.426874

48 hpi — Blood
Dox values: [ 58. 447. 331. 183. 539.  41. 105. 148. 393. 111.  66. 191. 193.]
  n=13, mean=215.846, median=183.000
noDox values: [405. 638. 748. 475. 389. 267. 373. 548. 480. 207. 555. 407.]
  n=12, mean=457.667, median=441.000
Mann–Whitney: U=20, p=0.00176247
Welch: t=-3.87363, p=0.000770077

Summary (two-sided, unadjusted p-values)
        case  n_Dox  n_n